# 4. Base versus Post-trained Models

The previous notebooks used `Qwen3-0.6B-Base`, whose pretraining objective is next-token prediction over a broad corpus. We now compare it with `Qwen3-0.6B`, which starts from the base model and receives additional post-training intended to make it follow instructions and participate in conversations.

We will separate three things that are easy to mix together: the learned weights, the text format presented to the model and the decoding strategy used to select output tokens. A chat template can mark roles such as system, user and assistant, but formatting alone does not teach a base model to follow instructions. Conversely, evaluating a post-trained model without its expected format can hide behavior that its weights learned.

This is a small controlled qualitative experiment rather than a benchmark. We keep the model scale, prompt tasks and generation settings aligned where possible, record counterexamples and avoid treating a few responses as a universal measurement of either model.

## 1. Model pair and experiment setup

We pin both repositories to immutable commits so that a future run cannot silently receive different files from `main`. The [base model card](https://huggingface.co/Qwen/Qwen3-0.6B-Base/blob/da87bfb608c14b7cf20ba1ce41287e8de496c0cd/README.md) describes the pretrained checkpoint, while the [post-trained model card](https://huggingface.co/Qwen/Qwen3-0.6B/blob/167b8104f88905a951069f5f95f9776908da5f68/README.md) documents the conversational checkpoint and its thinking controls.

By default this completed experiment reuses the pinned local snapshots without network access. Set `LOCAL_FILES_ONLY=False` to fetch missing metadata and verify revisions against the Hub. The metadata download excludes weights. We load both configurations and tokenizers from their resolved local snapshots, and compare the fields relevant to this experiment. Identical architecture does not imply identical behavior: post-training primarily changes the learned parameter values, which are not described by dimensions such as layer count or hidden size.

Later we will load one checkpoint at a time. This keeps peak memory close to the cost of one model and makes the lifecycle explicit: load, generate, retain only the textual results, then release the model before loading the other checkpoint.

In [1]:
import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import torch
from huggingface_hub import HfApi, snapshot_download
from transformers import AutoConfig, AutoTokenizer

MODEL_SPECS = {
    "base": {
        "model_id": "Qwen/Qwen3-0.6B-Base",
        "revision": "da87bfb608c14b7cf20ba1ce41287e8de496c0cd",
    },
    "post_trained": {
        "model_id": "Qwen/Qwen3-0.6B",
        "revision": "167b8104f88905a951069f5f95f9776908da5f68",
    },
}
METADATA_PATTERNS = (
    "README.md",
    "config.json",
    "generation_config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "added_tokens.json",
)
CONFIG_FIELDS = (
    "architectures",
    "vocab_size",
    "hidden_size",
    "intermediate_size",
    "num_hidden_layers",
    "num_attention_heads",
    "num_key_value_heads",
    "head_dim",
    "max_position_embeddings",
    "tie_word_embeddings",
    "bos_token_id",
    "eos_token_id",
)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    MODEL_DTYPE = (
        torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    )
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    MODEL_DTYPE = torch.bfloat16
else:
    DEVICE = torch.device("cpu")
    MODEL_DTYPE = torch.float32

LOCAL_FILES_ONLY = True  # Both pinned checkpoints were downloaded in earlier runs.
api = HfApi()
configs = {}
tokenizers = {}
model_metadata = {}

for label, spec in MODEL_SPECS.items():
    resolved_revision = spec["revision"]
    if not LOCAL_FILES_ONLY:
        resolved_revision = api.model_info(spec["model_id"], revision=spec["revision"]).sha
    if resolved_revision != spec["revision"]:
        raise RuntimeError(
            f"Resolved {label} revision {resolved_revision}, expected {spec['revision']}"
        )

    snapshot_path = Path(snapshot_download(
        repo_id=spec["model_id"],
        revision=spec["revision"],
        allow_patterns=METADATA_PATTERNS,
        local_files_only=LOCAL_FILES_ONLY,
    ))
    config = AutoConfig.from_pretrained(snapshot_path, local_files_only=True)
    tokenizer = AutoTokenizer.from_pretrained(
        snapshot_path, local_files_only=True
    )
    configs[label] = config
    tokenizers[label] = tokenizer

    tokenizer_path = snapshot_path / "tokenizer.json"
    chat_template = tokenizer.chat_template or ""
    if not isinstance(chat_template, str):
        chat_template = json.dumps(chat_template, sort_keys=True)

    model_metadata[label] = {
        "model_id": spec["model_id"],
        "revision": resolved_revision,
        "snapshot_path": str(snapshot_path),
        "model_card": (
            f"https://huggingface.co/{spec['model_id']}/blob/{resolved_revision}/README.md"
        ),
        "config": {field: getattr(config, field) for field in CONFIG_FIELDS},
        "tokenizer": {
            "class": type(tokenizer).__name__,
            "base_vocab_size": tokenizer.vocab_size,
            "total_tokens": len(tokenizer),
            "model_max_length": tokenizer.model_max_length,
            "special_tokens_map": tokenizer.special_tokens_map,
            "tokenizer_sha256": hashlib.sha256(
                tokenizer_path.read_bytes()
            ).hexdigest(),
            "has_chat_template": bool(chat_template),
            "chat_template_sha256": hashlib.sha256(
                chat_template.encode("utf-8")
            ).hexdigest(),
        },
    }

environment = {
    "started_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version.split()[0],
    "os": platform.platform(),
    "packages": {
        name: version(name)
        for name in ("torch", "transformers", "huggingface-hub")
    },
    "device": str(DEVICE),
    "local_files_only": LOCAL_FILES_ONLY,
    "runtime_dtype": str(MODEL_DTYPE),
}
print("Environment:")
print(json.dumps(environment, indent=2, ensure_ascii=False))
print("\nPinned model metadata:")
print(json.dumps(model_metadata, indent=2, ensure_ascii=False))

for section in ("config", "tokenizer"):
    base_facts = model_metadata["base"][section]
    post_facts = model_metadata["post_trained"][section]
    differences = {
        key: {"base": base_facts[key], "post_trained": post_facts[key]}
        for key in base_facts
        if base_facts[key] != post_facts[key]
    }
    print(f"\nSelected {section} differences:")
    print(
        json.dumps(differences, indent=2, ensure_ascii=False)
        if differences
        else "None in the selected fields."
    )

Environment:
{
  "started_utc": "2026-11-02T15:54:27.723353+00:00",
  "python": "3.12.13",
  "os": "macOS-26.6.2-arm64-arm-64bit",
  "packages": {
    "torch": "2.11.0",
    "transformers": "5.19.0",
    "huggingface-hub": "2.2.0"
  },
  "device": "mps",
  "local_files_only": true,
  "runtime_dtype": "torch.bfloat16"
}

Pinned model metadata:
{
  "base": {
    "model_id": "Qwen/Qwen3-0.6B-Base",
    "revision": "da87bfb608c14b7cf20ba1ce41287e8de496c0cd",
    "snapshot_path": "/Users/sergiogomez/.cache/huggingface/hub/models--Qwen--Qwen3-0.6B-Base/snapshots/da87bfb608c14b7cf20ba1ce41287e8de496c0cd",
    "model_card": "https://huggingface.co/Qwen/Qwen3-0.6B-Base/blob/da87bfb608c14b7cf20ba1ce41287e8de496c0cd/README.md",
    "config": {
      "architectures": [
        "Qwen3ForCausalLM"
      ],
      "vocab_size": 151936,
      "hidden_size": 1024,
      "intermediate_size": 3072,
      "num_hidden_layers": 28,
      "num_attention_heads": 16,
      "num_key_value_heads": 8,
      "head_

## 2. Create a fixed prompt set

The selected configuration fields confirm that both checkpoints use the same transformer dimensions. The pinned post-trained config declares a longer positional limit (`40,960` versus `32,768`) and uses `<|im_end|>` as EOS instead of `<|endoftext|>`. Both tokenizers expose the same chat-template hash, which is useful evidence that possessing a template and learning to respond to it are separate facts.

Before generating anything, we define five fixed tasks covering continuation, factual recall, instruction following, strict structured output and Spanish. We record qualitative predictions in advance so that we do not rewrite our expectations after seeing the outputs. The task text will remain unchanged between checkpoints; only the appropriate input format will differ later: plain text for the base model and chat messages for the post-trained model.

We use deterministic greedy decoding for the main comparison. This isolates the checkpoint and input format from sampling noise. It does not identify the best production settings or measure general model quality, but it makes every result reproducible within the numerical limits of the device.

In [2]:
PROMPT_CASES = [
    {
        "id": "continuation",
        "category": "text continuation",
        "prompt": (
            "The lighthouse keeper opened the old logbook and discovered"
        ),
        "base_prediction": "Continue the unfinished sentence as ordinary text.",
        "post_prediction": (
            "React to the fragment as a user message; it may continue it or ask what is wanted."
        ),
    },
    {
        "id": "factual",
        "category": "factual question",
        "prompt": "What is the capital of Australia?",
        "base_prediction": (
            "May answer Canberra because question-answer patterns occur in pretraining, "
            "but is not explicitly optimized to answer."
        ),
        "post_prediction": "Answer Canberra directly as an assistant.",
    },
    {
        "id": "instruction",
        "category": "instruction following",
        "prompt": "Explain in exactly two sentences why the sky appears blue.",
        "base_prediction": (
            "May produce a plausible explanation without reliably respecting two sentences."
        ),
        "post_prediction": "Explain the phenomenon and respect the two-sentence constraint.",
    },
    {
        "id": "structured",
        "category": "structured output",
        "prompt": (
            "Return exactly one JSON object with the keys language and creator for "
            "Python. Do not add any other text."
        ),
        "base_prediction": (
            "May imitate JSON but is less likely to satisfy every formatting constraint."
        ),
        "post_prediction": "Return only a valid JSON object with both requested keys.",
    },
    {
        "id": "spanish",
        "category": "Spanish instruction",
        "prompt": (
            "Explica en español y en una sola frase qué es el aprendizaje automático."
        ),
        "base_prediction": (
            "May continue in Spanish but need not respect the one-sentence constraint."
        ),
        "post_prediction": "Answer in Spanish using one sentence.",
    },
]

GENERATION_SETTINGS = {
    "max_new_tokens": 64,
    "do_sample": False,
    "num_beams": 1,
    "use_cache": True,
}

assert len({case["id"] for case in PROMPT_CASES}) == len(PROMPT_CASES)

print("Fixed generation settings:")
print(json.dumps(GENERATION_SETTINGS, indent=2))
for case in PROMPT_CASES:
    print(f"\n[{case['id']}] {case['category']}")
    print(f"Prompt: {case['prompt']}")
    print(f"Base prediction: {case['base_prediction']}")
    print(f"Post-trained prediction: {case['post_prediction']}")


Fixed generation settings:
{
  "max_new_tokens": 64,
  "do_sample": false,
  "num_beams": 1,
  "use_cache": true
}

[continuation] text continuation
Prompt: The lighthouse keeper opened the old logbook and discovered
Base prediction: Continue the unfinished sentence as ordinary text.
Post-trained prediction: React to the fragment as a user message; it may continue it or ask what is wanted.

[factual] factual question
Prompt: What is the capital of Australia?
Base prediction: May answer Canberra because question-answer patterns occur in pretraining, but is not explicitly optimized to answer.
Post-trained prediction: Answer Canberra directly as an assistant.

[instruction] instruction following
Prompt: Explain in exactly two sentences why the sky appears blue.
Base prediction: May produce a plausible explanation without reliably respecting two sentences.
Post-trained prediction: Explain the phenomenon and respect the two-sentence constraint.

[structured] structured output
Prompt: Return

## 3. Base-model completions

Our predictions are hypotheses, not expected-answer labels. We now load only the base checkpoint from its existing local cache and present every task as plain text without role markers. This matches its core pretraining interface: a prefix enters the causal language model and it continues that token sequence.

A base model can still answer questions or follow instructions because its pretraining corpus contains many examples with those patterns. The important distinction is behavioral reliability: producing an answer once does not mean that the model was optimized to interpret arbitrary user text as a request. We preserve every continuation, including awkward or incorrect ones, rather than selecting favorable examples.

After generation, we retain only Python metadata, token IDs and decoded strings. We then delete the model and empty the accelerator allocator so that the post-trained checkpoint can later use the same memory. The cached weight file remains on disk; releasing the in-memory model does not delete it.

In [3]:
import gc

from transformers import AutoModelForCausalLM


def allocated_device_bytes():
    if DEVICE.type == "cuda":
        return torch.cuda.memory_allocated()
    if DEVICE.type == "mps":
        return torch.mps.current_allocated_memory()
    return None


base_spec = MODEL_SPECS["base"]
base_tokenizer = tokenizers["base"]
memory_before_load = allocated_device_bytes()
base_model = AutoModelForCausalLM.from_pretrained(
    base_spec["model_id"],
    revision=base_spec["revision"],
    local_files_only=True,
    dtype=MODEL_DTYPE,
).to(DEVICE)
base_model.eval()
memory_after_load = allocated_device_bytes()

parameter_count = sum(parameter.numel() for parameter in base_model.parameters())
parameter_bytes = sum(
    parameter.numel() * parameter.element_size()
    for parameter in base_model.parameters()
)
base_results = []

for case in PROMPT_CASES:
    encoded = base_tokenizer(
        case["prompt"],
        add_special_tokens=False,
        return_tensors="pt",
    ).to(DEVICE)
    prompt_length = encoded["input_ids"].shape[1]

    with torch.inference_mode():
        generated = base_model.generate(
            **encoded,
            **GENERATION_SETTINGS,
            eos_token_id=base_model.config.eos_token_id,
            pad_token_id=base_tokenizer.pad_token_id,
        )

    new_token_ids = generated[0, prompt_length:].cpu().tolist()
    full_token_ids = generated[0].cpu().tolist()
    base_results.append({
        "task_id": case["id"],
        "category": case["category"],
        "model_id": base_spec["model_id"],
        "revision": base_spec["revision"],
        "input_format": "plain text",
        "prompt": case["prompt"],
        "prompt_token_ids": encoded["input_ids"][0].cpu().tolist(),
        "generated_token_ids": new_token_ids,
        "generated_tokens": len(new_token_ids),
        "stopped_on_eos": bool(
            new_token_ids
            and new_token_ids[-1] == base_model.config.eos_token_id
        ),
        "response": base_tokenizer.decode(
            new_token_ids, skip_special_tokens=True
        ),
        "full_text": base_tokenizer.decode(
            full_token_ids, skip_special_tokens=True
        ),
        "prediction": case["base_prediction"],
        "settings": dict(GENERATION_SETTINGS),
    })

base_run_metadata = {
    "parameter_count": parameter_count,
    "parameter_storage_gib": parameter_bytes / 2**30,
    "device_memory_before_load_gib": (
        memory_before_load / 2**30 if memory_before_load is not None else None
    ),
    "device_memory_after_load_gib": (
        memory_after_load / 2**30 if memory_after_load is not None else None
    ),
}

print("Base run metadata:")
print(json.dumps(base_run_metadata, indent=2))
for result in base_results:
    print(
        f"\n[{result['task_id']}] tokens={result['generated_tokens']} "
        f"EOS={result['stopped_on_eos']}"
    )
    print(f"Prompt: {result['prompt']}")
    print(f"Continuation: {result['response']}")

del base_model, encoded, generated, new_token_ids
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
elif DEVICE.type == "mps":
    torch.mps.empty_cache()

memory_after_release = allocated_device_bytes()
print(
    "\nDevice memory after release:",
    (
        f"{memory_after_release / 2**30:.3f} GiB"
        if memory_after_release is not None
        else "not available on CPU"
    ),
)


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Base run metadata:
{
  "parameter_count": 596049920,
  "parameter_storage_gib": 1.1102294921875,
  "device_memory_before_load_gib": 0.0,
  "device_memory_after_load_gib": 1.1102299690246582
}

[continuation] tokens=64 EOS=False
Prompt: The lighthouse keeper opened the old logbook and discovered
Continuation:  that the number of days in the month of February in the year 1999 was 29. He then calculated the number of days in the month of February in the year 2000. What was the number of days in February in the year 2000?
To determine the

[factual] tokens=64 EOS=False
Prompt: What is the capital of Australia?
Continuation:  The capital of Australia is Canberra. It was chosen as the capital in 1908 after a series of debates and discussions among the Australian government and the public. Canberra is located in the state of New South Wales and is known for its modern architecture and as a symbol of the country's commitment to democracy and international

[instruction] tokens=64 EOS=False
Pro

## 4. Inspect the chat template

The base outputs show why related content is not the same as reliable instruction following: the model answered the factual question but added an error, ignored sentence and JSON constraints, and treated the Spanish request as a pattern to continue. We now inspect the format that the post-trained checkpoint expects before loading its weights.

A chat template is a Jinja program stored with the tokenizer. It converts structured messages such as `{role: user, content: ...}` into one flat token sequence because the causal model still receives only token IDs. Role delimiters mark where each turn begins and ends. `add_generation_prompt=True` appends the beginning of an assistant turn, placing the model at the exact position where its response should start.

For the main comparison we will set `enable_thinking=False`. Qwen's template expresses that choice in the rendered prefix; it does not modify the weights at runtime. We render one conversation with and without the assistant generation prefix, inspect the added suffix and verify two equivalent tokenization paths. Both checkpoints contain the same template hash, but only the post-trained weights were optimized to respond reliably to this conversational structure.

In [4]:
template_tokenizer = tokenizers["post_trained"]
TEMPLATE_MESSAGES = [
    {
        "role": "system",
        "content": "You are a concise and accurate assistant.",
    },
    {
        "role": "user",
        "content": "What is the capital of Australia?",
    },
]

rendered_conversation = template_tokenizer.apply_chat_template(
    TEMPLATE_MESSAGES,
    tokenize=False,
    add_generation_prompt=False,
    enable_thinking=False,
)
rendered_for_generation = template_tokenizer.apply_chat_template(
    TEMPLATE_MESSAGES,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
direct_template_encoding = template_tokenizer.apply_chat_template(
    TEMPLATE_MESSAGES,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,
)
direct_template_ids = direct_template_encoding["input_ids"]
render_then_tokenize_ids = template_tokenizer(
    rendered_for_generation,
    add_special_tokens=False,
)["input_ids"]

assert direct_template_ids == render_then_tokenize_ids
assert rendered_for_generation.startswith(rendered_conversation)
generation_suffix = rendered_for_generation[len(rendered_conversation):]
template_pieces = template_tokenizer.convert_ids_to_tokens(
    direct_template_ids
)

special_markers = (
    "<|im_start|>",
    "<|im_end|>",
    "<|endoftext|>",
    "<think>",
    "</think>",
)
special_marker_ids = {
    marker: template_tokenizer.convert_tokens_to_ids(marker)
    for marker in special_markers
}

print("Messages:")
print(json.dumps(TEMPLATE_MESSAGES, indent=2, ensure_ascii=False))
print(f"\nRendered conversation without generation prompt:\n{rendered_conversation!r}")
print(f"\nSuffix added for assistant generation:\n{generation_suffix!r}")
print(f"\nComplete rendered input:\n{rendered_for_generation!r}")
print(f"\nSpecial marker IDs: {special_marker_ids}")
print(f"Template token count: {len(direct_template_ids)}")
print("Tokenization paths match: True")
print(f"\n{'Pos':>3} {'ID':>8}  Token piece")
for position, (token_id, piece) in enumerate(
    zip(direct_template_ids, template_pieces, strict=True)
):
    print(f"{position:>3} {token_id:>8}  {piece!r}")


Messages:
[
  {
    "role": "system",
    "content": "You are a concise and accurate assistant."
  },
  {
    "role": "user",
    "content": "What is the capital of Australia?"
  }
]

Rendered conversation without generation prompt:
'<|im_start|>system\nYou are a concise and accurate assistant.<|im_end|>\n<|im_start|>user\nWhat is the capital of Australia?<|im_end|>\n'

Suffix added for assistant generation:
'<|im_start|>assistant\n<think>\n\n</think>\n\n'

Complete rendered input:
'<|im_start|>system\nYou are a concise and accurate assistant.<|im_end|>\n<|im_start|>user\nWhat is the capital of Australia?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

Special marker IDs: {'<|im_start|>': 151644, '<|im_end|>': 151645, '<|endoftext|>': 151643, '<think>': 151667, '</think>': 151668}
Template token count: 32
Tokenization paths match: True

Pos       ID  Token piece
  0   151644  '<|im_start|>'
  1     8948  'system'
  2      198  'Ċ'
  3     2610  'You'
  4      525  'Ġare'
  

## 5. Post-trained model responses

The template inspection exposed the complete input protocol. `add_generation_prompt=True` opened an assistant turn, and `enable_thinking=False` inserted an already closed, empty `<think>...</think>` block so generation begins in the final-answer region. The result was still an ordinary sequence of 32 token IDs; roles have meaning because the post-trained weights learned their pattern, not because the forward pass receives a special role argument.

We now load the pinned post-trained checkpoint (from the local cache by default). Each fixed task becomes one user message, with no extra system instruction, so its content remains identical to the plain text received by the base model. The chat template necessarily adds role markers and the non-thinking prefix. That is the model's intended interface, but it also means the main comparison changes both weights and input formatting; we will isolate formatting in the following section.

We retain the same deterministic decoding settings and 64-token budget. We store both the clean response and the raw decoded continuation containing any special markers. Unlike the base checkpoint, this model uses `<|im_end|>` as EOS, allowing it to signal that its assistant turn is complete. We keep the model in memory after this cell because the next controls use the same weights.

In [5]:
post_spec = MODEL_SPECS["post_trained"]
post_tokenizer = tokenizers["post_trained"]
memory_before_post_load = allocated_device_bytes()
post_model = AutoModelForCausalLM.from_pretrained(
    post_spec["model_id"],
    revision=post_spec["revision"],
    dtype=MODEL_DTYPE,
    local_files_only=LOCAL_FILES_ONLY,
).to(DEVICE)
post_model.eval()
memory_after_post_load = allocated_device_bytes()

post_parameter_count = sum(
    parameter.numel() for parameter in post_model.parameters()
)
post_parameter_bytes = sum(
    parameter.numel() * parameter.element_size()
    for parameter in post_model.parameters()
)
post_results = []

for case in PROMPT_CASES:
    messages = [{"role": "user", "content": case["prompt"]}]
    rendered_input = post_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    encoded = post_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
    ).to(DEVICE)
    prompt_length = encoded["input_ids"].shape[1]

    with torch.inference_mode():
        generated = post_model.generate(
            **encoded,
            **GENERATION_SETTINGS,
            eos_token_id=post_model.config.eos_token_id,
            pad_token_id=post_tokenizer.pad_token_id,
        )

    new_token_ids = generated[0, prompt_length:].cpu().tolist()
    post_results.append({
        "task_id": case["id"],
        "category": case["category"],
        "model_id": post_spec["model_id"],
        "revision": post_spec["revision"],
        "input_format": "chat template, non-thinking",
        "prompt": case["prompt"],
        "rendered_input": rendered_input,
        "prompt_token_ids": encoded["input_ids"][0].cpu().tolist(),
        "generated_token_ids": new_token_ids,
        "generated_tokens": len(new_token_ids),
        "stopped_on_eos": bool(
            new_token_ids
            and new_token_ids[-1] == post_model.config.eos_token_id
        ),
        "response": post_tokenizer.decode(
            new_token_ids, skip_special_tokens=True
        ).strip(),
        "raw_response": post_tokenizer.decode(
            new_token_ids, skip_special_tokens=False
        ),
        "prediction": case["post_prediction"],
        "settings": {
            **GENERATION_SETTINGS,
            "enable_thinking": False,
        },
    })

post_run_metadata = {
    "parameter_count": post_parameter_count,
    "parameter_storage_gib": post_parameter_bytes / 2**30,
    "device_memory_before_load_gib": (
        memory_before_post_load / 2**30
        if memory_before_post_load is not None
        else None
    ),
    "device_memory_after_load_gib": (
        memory_after_post_load / 2**30
        if memory_after_post_load is not None
        else None
    ),
    "eos_token": post_tokenizer.eos_token,
    "eos_token_id": post_model.config.eos_token_id,
}

print("Post-trained run metadata:")
print(json.dumps(post_run_metadata, indent=2, ensure_ascii=False))
for result in post_results:
    print(
        f"\n[{result['task_id']}] tokens={result['generated_tokens']} "
        f"EOS={result['stopped_on_eos']}"
    )
    print(f"Prompt: {result['prompt']}")
    print(f"Response: {result['response']}")
    print(f"Raw ending: {result['raw_response'][-40:]!r}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Post-trained run metadata:
{
  "parameter_count": 596049920,
  "parameter_storage_gib": 1.1102294921875,
  "device_memory_before_load_gib": 0.0,
  "device_memory_after_load_gib": 1.1102299690246582,
  "eos_token": "<|im_end|>",
  "eos_token_id": 151645
}

[continuation] tokens=64 EOS=False
Prompt: The lighthouse keeper opened the old logbook and discovered
Response: The lighthouse keeper opened the old logbook and discovered a series of entries that described the lighthouse's daily operations, including the maintenance of the beacon, the crew's schedule, and the weather conditions. The logbook contained detailed information about the lighthouse's history, its location, and its significance in the region.
Raw ending: 'ion, and its significance in the region.'

[factual] tokens=11 EOS=True
Prompt: What is the capital of Australia?
Response: The capital of Australia is **Canberra**.
Raw ending: ' of Australia is **Canberra**.<|im_end|>'

[instruction] tokens=56 EOS=True
Prompt: Explain in

### Try the post-trained model

In [6]:
MY_PROMPT = "Explain KV caching as if I already understand self-attention."
MY_SYSTEM_PROMPT = "Hi manin"
MY_ENABLE_THINKING = False
MY_MAX_NEW_TOKENS = 128
MY_DO_SAMPLE = False
MY_TEMPERATURE = 0.7
MY_TOP_P = 0.8
MY_TOP_K = 20
MY_SEED = 123

my_messages = []
if MY_SYSTEM_PROMPT:
    my_messages.append({"role": "system", "content": MY_SYSTEM_PROMPT})
my_messages.append({"role": "user", "content": MY_PROMPT})

my_rendered_input = post_tokenizer.apply_chat_template(
    my_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=MY_ENABLE_THINKING,
)
my_inputs = post_tokenizer.apply_chat_template(
    my_messages,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=MY_ENABLE_THINKING,
    return_tensors="pt",
).to(DEVICE)

my_generation_settings = {
    "max_new_tokens": MY_MAX_NEW_TOKENS,
    "do_sample": MY_DO_SAMPLE,
    "use_cache": True,
    "eos_token_id": post_model.config.eos_token_id,
    "pad_token_id": post_tokenizer.pad_token_id,
}
if MY_DO_SAMPLE:
    my_generation_settings.update({
        "temperature": MY_TEMPERATURE,
        "top_p": MY_TOP_P,
        "top_k": MY_TOP_K,
    })

torch.manual_seed(MY_SEED)
with torch.inference_mode():
    my_generated = post_model.generate(
        **my_inputs,
        **my_generation_settings,
    )

my_prompt_length = my_inputs["input_ids"].shape[1]
my_new_token_ids = my_generated[0, my_prompt_length:].cpu().tolist()
my_response = post_tokenizer.decode(
    my_new_token_ids, skip_special_tokens=True
).strip()
my_stopped_on_eos = bool(
    my_new_token_ids
    and my_new_token_ids[-1] == post_model.config.eos_token_id
)

print(f"Rendered input:\n{my_rendered_input}")
print(
    f"Generated tokens: {len(my_new_token_ids)} | "
    f"EOS: {my_stopped_on_eos} | sampling: {MY_DO_SAMPLE}"
)
print(f"\nResponse:\n{my_response}")


Rendered input:
<|im_start|>system
Hi manin<|im_end|>
<|im_start|>user
Explain KV caching as if I already understand self-attention.<|im_end|>
<|im_start|>assistant
<think>

</think>


Generated tokens: 128 | EOS: False | sampling: False

Response:
Sure! Let's break down **KV caching** in the context of **self-attention** (like in transformers) to help you understand it better.

---

### 🔍 What is KV Caching?

In self-attention mechanisms, we often use **queries (Q)**, **keys (K)**, and **values (V)** to compute attention weights. The standard formula for self-attention is:

$$
\text{Attention} = \frac{1}{\text{d}} \cdot \text{Q} \cdot \text{K}^T \cdot \text{V}
$$

Where:
-


## 6. Separate formatting from model behavior

We keep the post-trained weights, five tasks, greedy decoding and 64-token budget fixed. Only the input representation changes: plain text versus the non-thinking chat template. The template includes role delimiters, an assistant prefix and an empty thinking block, so this control measures their combined effect, not any individual marker.

Prediction: plain text may still elicit correct answers, but may continue a document or invent further turns. We reuse the chat results from section 5. This control does not isolate the weight-only effect of post-training; that would require comparing both checkpoints under an identical format as well.

In [7]:
def run_post_case(case, input_format, thinking=False, max_new_tokens=64):
    if input_format == "plain text":
        rendered = case["prompt"]
    else:
        rendered = post_tokenizer.apply_chat_template(
            [{"role": "user", "content": case["prompt"]}],
            tokenize=False, add_generation_prompt=True, enable_thinking=thinking,
        )
    # input_ids and attention_mask: [batch=1, prompt_length].
    inputs = post_tokenizer(rendered, add_special_tokens=False, return_tensors="pt").to(DEVICE)
    settings = {**GENERATION_SETTINGS, "max_new_tokens": max_new_tokens}
    with torch.inference_mode():
        output = post_model.generate(
            **inputs, **settings,
            eos_token_id=post_model.config.eos_token_id,
            pad_token_id=post_tokenizer.pad_token_id,
        )
    # generate returns [1, prompt_length + generated_length]; retain only new tokens.
    ids = output[0, inputs["input_ids"].shape[1]:].cpu().tolist()
    assert 0 < len(ids) <= max_new_tokens
    return {
        "task_id": case["id"], "model_id": post_spec["model_id"],
        "revision": post_spec["revision"], "input_format": input_format,
        "prompt": case["prompt"], "rendered_input": rendered,
        "prompt_token_ids": inputs["input_ids"][0].cpu().tolist(),
        "generated_token_ids": ids, "generated_tokens": len(ids),
        "stopped_on_eos": ids[-1] == post_model.config.eos_token_id,
        "response": post_tokenizer.decode(ids, skip_special_tokens=True).strip(),
        "raw_response": post_tokenizer.decode(ids, skip_special_tokens=False),
        "settings": {**settings, "enable_thinking": thinking if input_format != "plain text" else None},
    }

plain_post_results = [run_post_case(case, "plain text") for case in PROMPT_CASES]
for plain, chat in zip(plain_post_results, post_results, strict=True):
    assert plain["task_id"] == chat["task_id"]
    print(f"\n[{plain['task_id']}]")
    for result in (plain, chat):
        print(f"{result['input_format']} | tokens={result['generated_tokens']} | EOS={result['stopped_on_eos']}")
        print(result["response"])


[continuation]
plain text | tokens=64 | EOS=False
that the angle of elevation from the top of the lighthouse to the top of the building is 30 degrees. The distance from the base of the lighthouse to the base of the building is 100 meters. What is the height of the lighthouse?

To solve this problem, we can use the
chat template, non-thinking | tokens=64 | EOS=False
The lighthouse keeper opened the old logbook and discovered a series of entries that described the lighthouse's daily operations, including the maintenance of the beacon, the crew's schedule, and the weather conditions. The logbook contained detailed information about the lighthouse's history, its location, and its significance in the region.

[factual]
plain text | tokens=64 | EOS=False
The capital of Australia is Canberra. What is the capital of the United States? The capital of the United States is Washington, D.C. What is the capital of Japan? The capital of Japan is Tokyo. What is the capital of India? The capital of I

## 7. Thinking mode under a fixed budget

We compare thinking enabled and disabled on the factual and JSON tasks, keeping the same post-trained weights and greedy decoding. Both modes get 256 new tokens, separate from the main 64-token comparison. Prediction: thinking will spend more tokens before the final answer and may exhaust this small budget.

We inspect raw markers and count tokens before and after `</think>`. Text emitted in the thinking region is model-generated text, not a guaranteed faithful explanation of internal computation. If the closing marker never arrives, we do not label the unfinished text as a final answer. Greedy and this short budget are controlled experimental choices, not recommended settings or a test of maximum reasoning capability.

In [8]:
THINKING_BUDGET = 256
thinking_results = []
close_think_id = post_tokenizer.convert_tokens_to_ids("</think>")
for case in PROMPT_CASES:
    if case["id"] not in {"factual", "structured"}:
        continue
    for thinking in (False, True):
        result = run_post_case(case, "chat template", thinking, THINKING_BUDGET)
        ids = result["generated_token_ids"]
        closed = close_think_id in ids
        boundary = ids.index(close_think_id) + 1 if closed else len(ids)
        if thinking:
            final_ids = ids[boundary:] if closed else []
            result["thinking_region_tokens"] = boundary
            result["final_answer"] = post_tokenizer.decode(final_ids, skip_special_tokens=True).strip()
        else:
            result["thinking_region_tokens"] = 0
            result["final_answer"] = result["response"]
        result["thinking_closed"] = closed if thinking else None
        thinking_results.append(result)
        print(f"\n[{case['id']}] thinking={thinking} tokens={len(ids)} EOS={result['stopped_on_eos']} closed={result['thinking_closed']}")
        print("Raw output:", result["raw_response"])
        print("Final answer:", result["final_answer"] or "[not reached within budget]")


[factual] thinking=False tokens=11 EOS=True closed=None
Raw output: The capital of Australia is **Canberra**.<|im_end|>
Final answer: The capital of Australia is **Canberra**.



[factual] thinking=True tokens=155 EOS=True closed=True
Raw output: <think>
Okay, the user is asking about the capital of Australia. I need to make sure I recall the correct answer. Australia is a country in the southern hemisphere, and its capital is Canberra. I think it's a bit of a tricky question because sometimes people might confuse it with other countries. Let me double-check. Yes, Canberra is the capital. It's also known as the "City of Peace" because of its peaceful environment. I should mention that it's a city in the central part of Australia, not in the capital city of Sydney. That way, the user gets the correct information without confusion.
</think>

The capital of Australia is **Canberra**. It is located in the central part of the country and is known for its peaceful environment.<|im_end|>
Final answer: The capital of Australia is **Canberra**. It is located in the central part of the country and is known for its peaceful environment.



[structured] thinking=False tokens=21 EOS=True closed=None
Raw output: ```json
{
  "language": "Python",
  "creator": "None"
}
```<|im_end|>
Final answer: ```json
{
  "language": "Python",
  "creator": "None"
}
```



[structured] thinking=True tokens=256 EOS=False closed=False
Raw output: <think>
Okay, the user wants me to return exactly one JSON object with keys 'language' and 'creator' for Python. Let me think about how to approach this.

First, I need to make sure that the JSON object only has those two keys. The user specified "exactly one JSON object," so I can't include any other fields. Also, they mentioned not to add any other text, so I should just output the JSON without any explanation.

Now, the keys are 'language' and 'creator'. For the 'language' key, I should use the value 'Python'. The 'creator' key would be the name of the Python interpreter, which is usually something like 'Python 3.10' or 'Python'. But maybe the user expects a specific value. Wait, the user didn't specify the creator's name, so I should use a common one. Let me check if there's a standard. Oh, sometimes people use 'Python' as the language, but the creator might be the interpreter. Hmm, maybe the creator is the n

## 8. Evaluate and conclude

The table preserves every response and its provenance. A JSON check parses the entire response, without stripping Markdown fences: the prompt asked for JSON only. It checks the exact key set separately from factual correctness. Mentioning Canberra is only a lexical check, not proof that the whole response is accurate. Sentence count and explanation quality are assessed by reading the outputs rather than a fragile punctuation-based score.

`budget reached` means the run used its token budget without EOS; it does not by itself prove that the visible last sentence is incomplete. Full revisions and settings appear in the run records below. These five examples are a qualitative comparison, not a general benchmark.

In [9]:
from html import escape
from IPython.display import HTML, display

all_results = base_results + post_results + plain_post_results
assert len(all_results) == 3 * len(PROMPT_CASES)

def observed_checks(result):
    notes = ["EOS" if result["stopped_on_eos"] else "budget reached"]
    if result["task_id"] == "structured":
        try:
            value = json.loads(result["response"])
        except json.JSONDecodeError:
            notes.append("not standalone JSON")
        else:
            correct_keys = isinstance(value, dict) and set(value) == {"language", "creator"}
            notes.append(f"valid JSON; exact keys={correct_keys}")
    if result["task_id"] == "factual":
        notes.append(f"mentions Canberra={'canberra' in result['response'].lower()}")
    return "; ".join(notes)

rows = []
for result in all_results:
    result["observed_checks"] = observed_checks(result)
    cells = [result["task_id"], result["model_id"] + " @ " + result["revision"],
             result["input_format"], json.dumps(result["settings"]),
             result["response"], result["observed_checks"]]
    rows.append("<tr>" + "".join("<td style='vertical-align:top;white-space:pre-wrap'>" + escape(str(x)) + "</td>" for x in cells) + "</tr>")
display(HTML("<table><tr><th>Task</th><th>Checkpoint</th><th>Input format</th><th>Settings</th><th>Response</th><th>Observations</th></tr>" + "".join(rows) + "</table>"))
for result in all_results:
    print(result["task_id"], result["model_id"], result["input_format"], result["observed_checks"])

# Results contain CPU lists/strings only; release the final model and playground tensors.
del post_model, encoded, generated, my_inputs, my_generated
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
elif DEVICE.type == "mps":
    torch.mps.empty_cache()
print("Device bytes after release:", allocated_device_bytes())

Task,Checkpoint,Input format,Settings,Response,Observations
continuation,Qwen/Qwen3-0.6B-Base @ da87bfb608c14b7cf20ba1ce41287e8de496c0cd,plain text,"{""max_new_tokens"": 64, ""do_sample"": false, ""num_beams"": 1, ""use_cache"": true}",that the number of days in the month of February in the year 1999 was 29. He then calculated the number of days in the month of February in the year 2000. What was the number of days in February in the year 2000? To determine the,budget reached
factual,Qwen/Qwen3-0.6B-Base @ da87bfb608c14b7cf20ba1ce41287e8de496c0cd,plain text,"{""max_new_tokens"": 64, ""do_sample"": false, ""num_beams"": 1, ""use_cache"": true}",The capital of Australia is Canberra. It was chosen as the capital in 1908 after a series of debates and discussions among the Australian government and the public. Canberra is located in the state of New South Wales and is known for its modern architecture and as a symbol of the country's commitment to democracy and international,budget reached; mentions Canberra=True
instruction,Qwen/Qwen3-0.6B-Base @ da87bfb608c14b7cf20ba1ce41287e8de496c0cd,plain text,"{""max_new_tokens"": 64, ""do_sample"": false, ""num_beams"": 1, ""use_cache"": true}","The sky appears blue because of the way light interacts with molecules in the Earth's atmosphere. When sunlight enters the atmosphere, it is scattered by tiny particles and molecules, such as nitrogen and oxygen. This scattering process, known as Rayleigh scattering, causes shorter wavelengths of light (like blue) to scatter more than longer wavelengths",budget reached
structured,Qwen/Qwen3-0.6B-Base @ da87bfb608c14b7cf20ba1ce41287e8de496c0cd,plain text,"{""max_new_tokens"": 64, ""do_sample"": false, ""num_beams"": 1, ""use_cache"": true}","The language is Python. The creator is ""Python"". The creator is ""Python"". The creator is ""Python"". The creator is ""Python"". The creator is ""Python"". The creator is ""Python"". The creator is ""Python"". The creator is ""Python"". The creator is ""Python"". The creator is ""Python",budget reached; not standalone JSON
spanish,Qwen/Qwen3-0.6B-Base @ da87bfb608c14b7cf20ba1ce41287e8de496c0cd,plain text,"{""max_new_tokens"": 64, ""do_sample"": false, ""num_beams"": 1, ""use_cache"": true}",¿Qué es un modelo de aprendizaje automático? ¿Qué es un modelo de aprendizaje supervisado? ¿Qué es un modelo de aprendizaje no supervisado? ¿Qué es un modelo de aprendizaje de red neuronal? ¿Qué es un modelo de aprendizaje de red de,budget reached
continuation,Qwen/Qwen3-0.6B @ 167b8104f88905a951069f5f95f9776908da5f68,"chat template, non-thinking","{""max_new_tokens"": 64, ""do_sample"": false, ""num_beams"": 1, ""use_cache"": true, ""enable_thinking"": false}","The lighthouse keeper opened the old logbook and discovered a series of entries that described the lighthouse's daily operations, including the maintenance of the beacon, the crew's schedule, and the weather conditions. The logbook contained detailed information about the lighthouse's history, its location, and its significance in the region.",budget reached
factual,Qwen/Qwen3-0.6B @ 167b8104f88905a951069f5f95f9776908da5f68,"chat template, non-thinking","{""max_new_tokens"": 64, ""do_sample"": false, ""num_beams"": 1, ""use_cache"": true, ""enable_thinking"": false}",The capital of Australia is **Canberra**.,EOS; mentions Canberra=True
instruction,Qwen/Qwen3-0.6B @ 167b8104f88905a951069f5f95f9776908da5f68,"chat template, non-thinking","{""max_new_tokens"": 64, ""do_sample"": false, ""num_beams"": 1, ""use_cache"": true, ""enable_thinking"": false}","The sky appears blue because of the way light interacts with the Earth's atmosphere, where shorter wavelengths of light (blue) are scattered more efficiently than longer wavelengths (red and orange). This scattering causes the blue light to be scattered in all directions, making the sky appear blue.",EOS
structured,Qwen/Qwen3-0.6B @ 167b8104f88905a951069f5f95f9776908da5f68,"chat template, non-thinking","{""max_new_tokens"

continuation Qwen/Qwen3-0.6B-Base plain text budget reached
factual Qwen/Qwen3-0.6B-Base plain text budget reached; mentions Canberra=True
instruction Qwen/Qwen3-0.6B-Base plain text budget reached
structured Qwen/Qwen3-0.6B-Base plain text budget reached; not standalone JSON
spanish Qwen/Qwen3-0.6B-Base plain text budget reached
continuation Qwen/Qwen3-0.6B chat template, non-thinking budget reached
factual Qwen/Qwen3-0.6B chat template, non-thinking EOS; mentions Canberra=True
instruction Qwen/Qwen3-0.6B chat template, non-thinking EOS
structured Qwen/Qwen3-0.6B chat template, non-thinking EOS; not standalone JSON
spanish Qwen/Qwen3-0.6B chat template, non-thinking EOS
continuation Qwen/Qwen3-0.6B plain text budget reached
factual Qwen/Qwen3-0.6B plain text budget reached; mentions Canberra=True
instruction Qwen/Qwen3-0.6B plain text budget reached
structured Qwen/Qwen3-0.6B plain text budget reached; not standalone JSON
spanish Qwen/Qwen3-0.6B plain text budget reached
Device bytes 

## Completion record

- [x] Implemented base/post-trained comparison and inspected the chat template.
- [x] Controlled input formatting with the same post-trained weights and decoding settings.
- [x] Compared thinking modes under an equal 256-token budget.
- [x] Executed all nine code cells in order from a fresh kernel on 2026-11-02, on Apple MPS with BF16, using pinned local checkpoints.
- [x] Preserved all outputs, recorded counterexamples and released model memory (0 allocated device bytes at the end).

## Experiment conclusions

### 1. Post-training changes behavior, not the next-token interface

Both checkpoints have 596,049,920 parameters and occupy about 1.110 GiB in BF16. Both still consume token IDs and predict the next token, just like mini GPT. In the intended-interface comparison, the base model continues text, while the post-trained model more often produces a bounded assistant response. However, this comparison changes weights AND formatting; it cannot assign the full difference to weights alone.

With the 64-token budget, all five base runs reached the cap without EOS. The post-trained chat runs emitted EOS in four of five cases. The sky explanation contained exactly two sentences; the Spanish response used one sentence but had grammatical problems. Ending a turn and following some constraints are useful behaviors, not guarantees of correctness.

### 2. The same weights behave differently with different input formats

The post-trained model in plain text answered Canberra, then continued with questions about other countries until the 64-token cap. With the non-thinking chat template it answered only the requested question and emitted EOS after 11 tokens. All five plain-text controls reached the cap without EOS. The Spanish plain-text prompt became a series of questions, whereas the chat input elicited an explanation.

Thus, the plain-text model did not lose its factual knowledge: the input changed which continuation pattern it followed. The template supplies learned conversational cues. This control changes the complete template, including the empty thinking block; it does not isolate individual role tokens.

### 3. Instruction following and factual accuracy are separate

The post-trained chat model returned a fenced JSON block with `"creator": "None"`. It failed both the request for standalone JSON (the entire response is not parseable JSON) and the factual value (Python's creator is Guido van Rossum). The plain-text post-trained run invented `John Doe`; the base run repeated `Python` as the creator. No main-comparison variant solved this task.

The base model named Canberra correctly but added the incorrect claim that it is in New South Wales. This illustrates why the lexical Canberra check in the table is not an accuracy score.

### 4. More thinking tokens did not imply a better answer

With the same 256-token cap in both modes, the factual task used 11 tokens without thinking and 155 with thinking. Thinking reached a final answer but added the inaccurate claim that Canberra is in the central part of Australia. For JSON, non-thinking finished in 21 tokens with the wrong fenced object; thinking exhausted 256 tokens before `</think>` and produced no final answer. We retain this as a budget-limited failure, not evidence that thinking can never help.

Thinking mode changes the prompt and subsequent generated sequence, not the weights. Its generated explanation is not a guarantee of faithful internal reasoning or factual checking.

### Limits and connection to the chapter

These are five hand-picked tasks, one small model pair, greedy decoding and short output budgets on one device/dtype. There is no general quality ranking or estimate of the causal effect of a specific post-training algorithm. Greedy eliminates sampling randomness but does not guarantee identical numerical results across hardware or software versions. Longer budgets, different prompts or sampling can change outcomes; they are future experiments, not requirements for closing this unit.

The chapter now connects the pieces: Hub repositories supply configuration, matching tokenizer and trained weights; a forward pass produces logits; a decoding loop turns logits into text; KV caching reuses past attention keys and values; and post-trained weights plus the expected input format orient that same loop toward assistant behavior. We used Transformers to inspect and compare a real checkpoint after building the core mechanism in mini GPT.
